In [8]:
import os
import json
import re
import pandas as pd

pd.options.display.max_columns = None

In [9]:
def extract_json(response: str):
    """Extract JSON content from a formatted string."""
    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        json_str = response.strip('```json').strip('```')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Chyba při dekódování JSON: {e}")
        return None

In [10]:
def process_txt_files(folder_path, prefix):
    """Zpracuje všechny txt soubory začínající prefixem (např. 'bank_part') ve složce a vrátí Pandas DataFrame."""
    all_data = []
    
    # Get files with prefix and end with .txt
    files = [f for f in os.listdir(folder_path) if f.startswith(prefix) and f.endswith(".txt")]
    
    # Order by number
    files.sort(key=lambda x: int(re.search(r'chunk(\d+)', x).group(1)))
    
    for filename in files:
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as file:
            for line in file:
                try:
                    json_obj = json.loads(line.strip())
                    content_str = json_obj.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
                    extracted_json = extract_json(content_str)
                    
                    if extracted_json:
                        row = {"id": json_obj["id"], "custom_id": json_obj["custom_id"]}
                        for feature in extracted_json.get("features", []):
                            row[feature["feature_name"]] = feature["answer"]
                        
                        all_data.append(row)
                except json.JSONDecodeError:
                    print(f"Chyba dekódování JSON v souboru {filename}")

    df = pd.DataFrame(all_data)
    return df

In [11]:
# Použití skriptu
folder_path = "../../data/outputs/c19"
df = process_txt_files(folder_path, "c19")

# Zobrazení výsledného dataframe
df

,id,custom_id,Research Topic,COVID-19 Relevance,Methodology Type,Sample Size,Geographical Focus,Funding Source,Publication Year,Journal Impact Factor,Author Affiliation,Data Type,Study Outcome,Ethical Considerations,Interdisciplinary Approach,Technological Innovation,Policy Implications,Clinical Trials,Statistical Analysis,Sample Demographics,Peer Review Status,Open Access
0,batch_req_67d53208f12481909eaae4c44e94729c,0,Sociology,No,Theoretical,Medium,North America,Non-profit,2023,Medium,University,Qualitative,Neutral,Yes,Yes,No,Yes,No,Descriptive,Gender,Peer-reviewed,No
1,batch_req_67d5320915248190988e5724e019e3d2,1,Medicine,No,Review,Medium,North America,Non-profit,2023,Medium,Hospital,Qualitative,Neutral,Yes,Yes,No,Yes,No,Descriptive,Ethnicity,Peer-reviewed,No
2,batch_req_67d532092fac8190b244d6d45a55a4a8,2,Public Health,Yes,Observational,Medium,Asia,Self-funded,2020,Medium,University,Quantitative,Negative,Yes,No,No,Yes,No,Descriptive,Age,Peer-reviewed,No
3,batch_req_67d53209440c81908a212a68bfcede16,3,Technology,No,Review,Medium,Global,Self-funded,2023,Medium,University,Qualitative,Neutral,No,No,Yes,No,No,Descriptive,Age,Non-peer-reviewed,No
4,batch_req_67d532095c648190acfffb6e36cd2ed7,4,Virology,Yes,Experimental,Small,Global,Not specified,2021,Not specified,Research Institute,Quantitative,Positive,Yes,Yes,No,No,No,Descriptive,Not specified,Peer-reviewed,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993,batch_req_67d55faacb6c8190a15e7d8d480ee9d1,2993,Technology,No,Experimental,Large,Global,University,2023,Medium,University,Quantitative,Positive,No,No,Yes,No,No,Descriptive,Age,Peer-reviewed,Yes
2994,batch_req_67d55faadc8c819082ca86fd9e565b53,2994,Public Health,Yes,Experimental,Medium,North America,Government,2020,Medium,University,Quantitative,Positive,Yes,Yes,No,Yes,No,Inferential,Age,Peer-reviewed,No
2995,batch_req_67d55faaff4c8190bbaee1d0c13b6b27,2995,Virology,No,Experimental,Medium,Global,Not specified,2023,Not specified,Not specified,Quantitative,Positive,No,No,No,No,No,Descriptive,Not specified,Not specified,Not specified
2996,batch_req_67d55fab12488190b7f643f33e53c268,2996,Immunology,No,Review,Medium,Global,Non-profit,2023,Medium,Research Institute,Qualitative,Neutral,Yes,No,No,Yes,No,Descriptive,Age,Peer-reviewed,No


# Merge with target

In [13]:
df['label'] = pd.read_csv("../../data/outputs/cord19/c19_final.csv")['cited']

In [14]:
df

,id,custom_id,Research Topic,COVID-19 Relevance,Methodology Type,Sample Size,Geographical Focus,Funding Source,Publication Year,Journal Impact Factor,Author Affiliation,Data Type,Study Outcome,Ethical Considerations,Interdisciplinary Approach,Technological Innovation,Policy Implications,Clinical Trials,Statistical Analysis,Sample Demographics,Peer Review Status,Open Access,label
0,batch_req_67d53208f12481909eaae4c44e94729c,0,Sociology,No,Theoretical,Medium,North America,Non-profit,2023,Medium,University,Qualitative,Neutral,Yes,Yes,No,Yes,No,Descriptive,Gender,Peer-reviewed,No,0
1,batch_req_67d5320915248190988e5724e019e3d2,1,Medicine,No,Review,Medium,North America,Non-profit,2023,Medium,Hospital,Qualitative,Neutral,Yes,Yes,No,Yes,No,Descriptive,Ethnicity,Peer-reviewed,No,0
2,batch_req_67d532092fac8190b244d6d45a55a4a8,2,Public Health,Yes,Observational,Medium,Asia,Self-funded,2020,Medium,University,Quantitative,Negative,Yes,No,No,Yes,No,Descriptive,Age,Peer-reviewed,No,0
3,batch_req_67d53209440c81908a212a68bfcede16,3,Technology,No,Review,Medium,Global,Self-funded,2023,Medium,University,Qualitative,Neutral,No,No,Yes,No,No,Descriptive,Age,Non-peer-reviewed,No,0
4,batch_req_67d532095c648190acfffb6e36cd2ed7,4,Virology,Yes,Experimental,Small,Global,Not specified,2021,Not specified,Research Institute,Quantitative,Positive,Yes,Yes,No,No,No,Descriptive,Not specified,Peer-reviewed,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993,batch_req_67d55faacb6c8190a15e7d8d480ee9d1,2993,Technology,No,Experimental,Large,Global,University,2023,Medium,University,Quantitative,Positive,No,No,Yes,No,No,Descriptive,Age,Peer-reviewed,Yes,1
2994,batch_req_67d55faadc8c819082ca86fd9e565b53,2994,Public Health,Yes,Experimental,Medium,North America,Government,2020,Medium,University,Quantitative,Positive,Yes,Yes,No,Yes,No,Inferential,Age,Peer-reviewed,No,1
2995,batch_req_67d55faaff4c8190bbaee1d0c13b6b27,2995,Virology,No,Experimental,Medium,Global,Not specified,2023,Not specified,Not specified,Quantitative,Positive,No,No,No,No,No,Descriptive,Not specified,Not specified,Not specified,1
2996,batch_req_67d55fab12488190b7f643f33e53c268,2996,Immunology,No,Review,Medium,Global,Non-profit,2023,Medium,Research Institute,Qualitative,Neutral,Yes,No,No,Yes,No,Descriptive,Age,Peer-reviewed,No,1


In [15]:
df = df.drop(columns=["id", "custom_id"])
data = pd.get_dummies( 
        df, sparse=False, prefix_sep='_'
    )

In [16]:
data

,label,Research Topic_Architecture,Research Topic_Biology,Research Topic_Economics,Research Topic_Education,Research Topic_Environmental Science,Research Topic_Epidemiology,Research Topic_Immunology,Research Topic_Medicine,Research Topic_Pediatrics,Research Topic_Philosophy,Research Topic_Psychology,Research Topic_Public Health,Research Topic_Sociology,Research Topic_Technology,Research Topic_Veterinary Medicine,Research Topic_Veterinary Science,Research Topic_Virology,COVID-19 Relevance_No,COVID-19 Relevance_Yes,Methodology Type_Case Study,Methodology Type_Experimental,Methodology Type_Historical,Methodology Type_Meta-analysis,Methodology Type_Mixed Methods,Methodology Type_Observational,Methodology Type_Qualitative,Methodology Type_Review,Methodology Type_Theoretical,Sample Size_Large,Sample Size_Medium,Sample Size_N/A,Sample Size_Small,Geographical Focus_Africa,Geographical Focus_Asia,Geographical Focus_Central America,Geographical Focus_Europe,Geographical Focus_Global,Geographical Focus_Local,Geographical Focus_North America,Geographical Focus_Oceania,Geographical Focus_South America,Funding Source_Government,Funding Source_Hospital,Funding Source_N/A,Funding Source_Non-profit,Funding Source_Not specified,Funding Source_Private,Funding Source_Self-funded,Funding Source_University,Publication Year_1991,Publication Year_2001,Publication Year_2002,Publication Year_2003,Publication Year_2004,Publication Year_2005,Publication Year_2006,Publication Year_2007,Publication Year_2008,Publication Year_2009,Publication Year_2010,Publication Year_2011,Publication Year_2012,Publication Year_2013,Publication Year_2014,Publication Year_2015,Publication Year_2016,Publication Year_2017,Publication Year_2018,Publication Year_2019,Publication Year_2020,Publication Year_2021,Publication Year_2022,Publication Year_2023,Publication Year_N/A,Publication Year_Not specified,Journal Impact Factor_High,Journal Impact Factor_Low,Journal Impact Factor_Medium,Journal Impact Factor_N/A,Journal Impact Factor_Not specified,Author Affiliation_Government Agency,Author Affiliation_Hospital,Author Affiliation_N/A,Author Affiliation_Non-profit,Author Affiliation_Not specified,Author Affiliation_Private Company,Author Affiliation_Research Institute,Author Affiliation_University,Data Type_Mixed Methods,Data Type_N/A,Data Type_Qualitative,Data Type_Quantitative,Data Type_Theoretical,Study Outcome_N/A,Study Outcome_Negative,Study Outcome_Neutral,Study Outcome_Positive,Ethical Considerations_N/A,Ethical Considerations_No,Ethical Considerations_Not specified,Ethical Considerations_Yes,Interdisciplinary Approach_No,Interdisciplinary Approach_Yes,Technological Innovation_No,Technological Innovation_Yes,Policy Implications_No,Policy Implications_Not specified,Policy Implications_Yes,Clinical Trials_No,Clinical Trials_Yes,Statistical Analysis_Descriptive,Statistical Analysis_Inferential,Statistical Analysis_N/A,Statistical Analysis_Not applicable,Statistical Analysis_Not specified,Statistical Analysis_Predictive,Statistical Analysis_Prescriptive,Sample Demographics_Age,"Sample Demographics_Age, Gender","Sample Demographics_Age, Gender, Ethnicity",Sample Demographics_Ethnicity,Sample Demographics_Gender,Sample Demographics_N/A,Sample Demographics_None,Sample Demographics_Not applicable,Sample Demographics_Not specified,Sample Demographics_Socioeconomic Status,Peer Review Status_N/A,Peer Review Status_Non-peer-reviewed,Peer Review Status_Not specified,Peer Review Status_Peer-reviewed,Open Access_N/A,Open Access_No,Open Access_Not specified,Open Access_Yes
0,0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,

In [17]:
# 1) Libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 2) Příprava feature matic X a cílové proměnné y
X = data.drop(columns=["label"])
y = data["label"]
#categorical_columns = X.select_dtypes(include=["object"]).columns
#X = df.drop(columns=categorical_columns)

# 3) Rozdělení na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)  # stratify, pokud je to klasifikace s nerovnoměrnými třídami

# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = RandomForestClassifier(random_state=42, class_weight='balanced')

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_grid = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=5,                  # 5-fold cross-validace
    scoring="accuracy",    # metrika, dle které se bude model porovnávat
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
y_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(y_test, y_pred))
print("Classification report na testu:")
print(classification_report(y_test, y_pred))
from sklearn.metrics import mean_absolute_error
print(f"MAE: {mean_absolute_error(y_test, y_pred)}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Nejlepší parametry: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
Nejlepší skóre na trénovací cross-validaci: 0.61758959638135
Přesnost na testu: 0.59
Classification report na testu:
              precision    recall  f1-score   support

           0       0.59      0.61      0.60       300
           1       0.59      0.57      0.58       300

    accuracy                           0.59       600
   macro avg       0.59      0.59      0.59       600
weighted avg       0.59      0.59      0.59       600

MAE: 0.41
